# 📘 Tutorial 2: Common PyTorch Tensor Operations

> This is the worked version of the notebook.
>
> All cells have been executed and include outputs, figures, and numerical results for reference.

This tutorial builds directly on the tensor foundations established in Tutorial 1 by introducing a broader set of common PyTorch operations used to **reshape, construct, combine, query, transform, and summarise tensors**.

Rather than treating these operations as isolated utility functions, the tutorial presents them as part of the practical working language of tensor computation. Once tensors are understood as structured mathematical objects, the next step is to learn how to manipulate them in ways that preserve clear dimensional meaning and computational clarity.

---

**The tutorial is designed to shift perspective**:
- from *“knowing what a tensor is”*
- to *“using tensors fluently through common PyTorch operations”*.

---

**The emphasis is on developing intuition for**:
- how tensor shape can be modified without changing underlying data,
- how useful structured tensors can be constructed efficiently,
- how tensors can be concatenated and queried,
- and how element-wise logic and statistical reductions operate in PyTorch.

---

**Key ideas explored include**:
- reshaping with `squeeze()` and `unsqueeze()`,
- constructing tensors with `torch.linspace()` and `torch.eye()`,
- combining tensors with `torch.cat()`,
- locating extrema with `torch.argmin()` and `torch.argmax()`,
- conditional selection with `torch.where()`,
- and statistical reduction with `torch.std()`.

---

**This tutorial serves as the practical foundation for**:
- autograd and computation graphs (Tutorial 3),
- gradient-based optimisation (later tutorials),
- and more advanced PyTorch workflows in which tensor operations become part of larger computational pipelines.

---

The focus is still intentionally not on neural networks, models, or training loops. Instead, the goal is to strengthen fluency with the kinds of tensor operations that appear constantly in real PyTorch code, so that later discussions of gradients and learning rest on a solid operational foundation.

---

**Recommended prerequisites**:
- Tutorial 1: PyTorch Tensor Fundamentals
- Basic familiarity with Python
- Elementary linear algebra (vectors and matrices)
- No prior experience with PyTorch automatic differentiation is assumed

---

**Author**: Angze Li

**Last updated**: 2026-04-14

**Version**: v1.0

## 🔧 Setup


In [1]:
import torch

## 1. Datashape and device

### float32

`float32` can represent:
- each number in the tensor is stored as a 32-bit floating-point number, aka **single precision**
- ~7 decimal digits of precision
- numbers up to ~10³⁸

float32 is the **default** dtype for PyTorch tensors, since float32 is fast on CPUs and GPUs, and accurate enough for most ML tasks.

---

### Device type

By default, PyTorch creates tensors on **cpu** because:
- CPUs are always available
- GPUs are optional
- moving data to GPU has overhead

---

We need to manually specify if we wish to create the tensor on GPU.

For **MacBook**, PyTorch can use **MPS (Metal Performance Shaders)**. We can check if MPS works by
```python
torch.backends.mps.is_available()
```
which would return either `True` or `False`. And if it is `True`, we *can* transfer the tensor to GPU.
```python
device = torch.device("mps")
x = x.to(device)
```
Similarly, for Windows/Linux platforms, we can check if cuda (NVIDIA GPUs) works by
```python
torch.backends.cuda.is_available()
```

---

But we almost always do not need to use GPU for computation unless **datasets get large or models are complex**.

It is also for this reason that we should have this setup at the start of a program:
```python
import torch
torch.set_default_dtype(torch.double)
device = torch.device("cpu")
```



In [2]:
tensor = torch.rand(3, 4)

print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Datatype of tensor: torch.float32
Device tensor is stored on: cpu


## 2. Squeeze and unsqueeze

### Removing dimensions with `squeeze()`

In many situations, tensors may contain **redundant dimensions of size 1**.
These often arise from data loading, batching, or reshaping operations.

The function `squeeze()` is used to **remove all dimensions of size 1** from a tensor.

In this example:

- The original tensor has shape `(1, 1, 4)`
  - two leading dimensions of size 1
- After applying `squeeze()`, these dimensions are removed
- The resulting tensor has shape `(4,)`

This operation does **not change the underlying data**, only how it is structured.

---

**Key idea**:
- `squeeze()` simplifies tensor shape by removing unnecessary singleton dimensions
- This is especially useful when preparing data for operations that expect a specific dimensionality

In [3]:
x = torch.tensor([[[1, 2, 3, 4]]], dtype=torch.float32)

print("Original tensor:")
print(x)
print("Shape:", x.shape)

x_squeezed = x.squeeze()

print("After squeeze():")
print(x_squeezed)
print("Shape:", x_squeezed.shape)

Original tensor:
tensor([[[1., 2., 3., 4.]]])
Shape: torch.Size([1, 1, 4])
After squeeze():
tensor([1., 2., 3., 4.])
Shape: torch.Size([4])


### Adding dimensions with `unsqueeze()`

While `squeeze()` removes unnecessary dimensions, `unsqueeze(dim)` does the opposite:
it **adds a new dimension of size 1 at a specified position**.

In this example, the original tensor has shape `(4,)`, i.e. a 1D tensor.

---

Applying `unsqueeze()` changes the shape depending on the value of `dim`:

- `unsqueeze(0)`
  → inserts a new dimension at position 0
  → shape becomes `(1, 4)`
  → interpretable as a **row vector**

- `unsqueeze(1)`
  → inserts a new dimension at position 1
  → shape becomes `(4, 1)`
  → interpretable as a **column vector**

- `unsqueeze(-1)`
  → inserts a new dimension at the **last position**
  → also results in shape `(4, 1)`
  → equivalent to `unsqueeze(1)` for a 1D tensor

---

**Key idea**:

The argument `dim` specifies **where the new axis is inserted**:

- `dim = 0` → before all existing dimensions
- `dim = 1` → after the first dimension
- `dim = -1` → at the end (last dimension)

---

This operation is important because many PyTorch operations expect tensors to have specific shapes.

For example:
- matrix multiplication distinguishes between row and column vectors
- batching often requires an extra leading dimension

So `unsqueeze()` is commonly used to **adapt tensor shapes without changing the underlying data**.

In [4]:
y = torch.tensor([1, 2, 3, 4], dtype=torch.float32)

print("Original tensor:")
print(y)
print("Shape:", y.shape)

y_unsqueezed0 = y.unsqueeze(0)
y_unsqueezed1 = y.unsqueeze(1)
y_unsqueezedm1 = y.unsqueeze(-1)

print("\nAfter unsqueeze(0):")
print(y_unsqueezed0)
print("Shape:", y_unsqueezed0.shape)

print("\nAfter unsqueeze(1):")
print(y_unsqueezed1)
print("Shape:", y_unsqueezed1.shape)

print("\nAfter unsqueeze(-1):")
print(y_unsqueezedm1)
print("Shape:", y_unsqueezedm1.shape)

Original tensor:
tensor([1., 2., 3., 4.])
Shape: torch.Size([4])

After unsqueeze(0):
tensor([[1., 2., 3., 4.]])
Shape: torch.Size([1, 4])

After unsqueeze(1):
tensor([[1.],
        [2.],
        [3.],
        [4.]])
Shape: torch.Size([4, 1])

After unsqueeze(-1):
tensor([[1.],
        [2.],
        [3.],
        [4.]])
Shape: torch.Size([4, 1])


## 3. Tensor construction, combination, and basic statistics

### Creating evenly spaced values with `torch.linspace()`

The function `torch.linspace(start, end, steps)` is used to create a 1D tensor of **evenly spaced values** between a specified range.

In this example:

- `start = 0` and `end = 1` define the interval
- `steps = 5` specifies how many values to generate
- The resulting tensor includes both the start and end points

So the output consists of 5 evenly spaced numbers from 0 to 1.

---

**Key idea**:

- `torch.linspace()` is useful for generating **structured numerical grids**
- It is commonly used in:
  - plotting functions,
  - parameter sweeps,
  - and constructing input ranges for mathematical operations

This provides a simple way to create controlled, evenly distributed data without manual specification.

In [5]:
x = torch.linspace(0, 1, steps=5)
print(x)

tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


### Creating an identity matrix with `torch.eye()`

The function `torch.eye(n)` creates an **identity matrix** of size $n \times n $.

An identity matrix has:
- **1s on the main diagonal**
- **0s everywhere else**

In this example:

- `torch.eye(4)` creates a $4 \times 4$ matrix
- The diagonal elements are 1, and all off-diagonal elements are 0

---

**Key idea**:

- Identity matrices act as the **multiplicative identity** in matrix algebra
  (i.e. $ A \cdot I = A $)
- They are commonly used in:
  - linear algebra operations,
  - initialisation schemes,
  - and constructing structured matrices

This makes `torch.eye()` a convenient way to generate matrices with a well-defined structure.

In [6]:
I = torch.eye(4)
print(I)

tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]])


### Joining tensors with `torch.cat()`

`torch.cat()` is used to **concatenate (join) multiple tensors along an existing dimension**.

A key requirement is that all tensors must have the **same shape in every dimension except the one you are concatenating along**.

---

**Understanding the `dim` argument**:

- `dim=0` → concatenate along the **first dimension**
  → increases the number of **rows** (vertical stacking)

- `dim=1` → concatenate along the **second dimension**
  → increases the number of **columns** (horizontal stacking)

- For higher-dimensional tensors:
  - `dim=2` → concatenate along the third dimension (often interpreted as features or channels)

- Negative indices count from the end:
  - `dim=-1` → last dimension
  - `dim=-2` → second-to-last dimension

---

**Intuition**:

Think of `torch.cat()` as *gluing tensors together along a chosen axis*:

- Along `dim=0` → stack tensors **on top of each other**
- Along `dim=1` → place tensors **side by side**

---

**Common pitfall**:

Concatenation will fail if shapes do not match in the non-concatenated dimensions.

For example:

- Valid: `(4, 4)` + `(4, 4)` along `dim=0` → `(8, 4)`
- Invalid: `(4, 4)` + `(3, 4)` along `dim=1` → mismatch in rows

---

**Key idea**:

`torch.cat()` does not create new values — it **reorganises existing data into a larger structured tensor**. Understanding how `dim` changes the structure is essential when building more complex tensor pipelines.### Joining tensors
`torch.cat` can be used to **concatenate** a sequence of tensors **along a given dimension**.
- Setting `dim=0` concatenates **rows** of the tensor.
- Setting `dim=1` concatenates **columns** of the tensor.
- If the tensor is 3D, setting `dim=2` concatenates "features" of the tensor.
- `dim=-1` refers to the last dimension of the tensor, and so on for `dim=-2`, `dim=-3`, etc.

In [7]:
# Note that these tensors are 2D
tensor1 = torch.ones(4, 4)
tensor0 = torch.zeros(4, 4)

t2 = torch.cat([tensor1, tensor0], dim=0)
print("Concatenation along dim=0:")
print(t2)
print("Shape:", t2.shape)

t1 = torch.cat([tensor1, tensor0], dim=1)
print("\nConcatenation along dim=1:")
print(t1)
print("Shape:", t1.shape)

Concatenation along dim=0:
tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
Shape: torch.Size([8, 4])

Concatenation along dim=1:
tensor([[1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.]])
Shape: torch.Size([4, 8])


### Measuring spread with `torch.std()`

The function `torch.std()` computes the **standard deviation** of a tensor, which measures how spread out the values are around the mean.

In this example, we apply `torch.std()` in three different ways:

---

- `torch.std(A)`
  → computes the standard deviation over **all elements** in the tensor
  → returns a **single scalar value**

- `torch.std(A, dim=0)`
  → computes the standard deviation **column-wise**
  → each column is treated as a separate set of values
  → returns a tensor containing one value per column

- `torch.std(A, dim=1)`
  → computes the standard deviation **row-wise**
  → each row is treated independently
  → returns a tensor containing one value per row

---

**Key idea**:

The `dim` argument specifies **which axis to reduce over**:

- `dim=0` → reduce across rows (compare values within each column)
- `dim=1` → reduce across columns (compare values within each row)

---

Standard deviation is an example of a **reduction operation**:
- it takes a tensor with many values
- and returns a smaller tensor (or scalar) that summarises those values

Understanding these reductions is important for analysing data and building statistical intuition in tensor-based workflows.

In [8]:
A = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])

print("A:")
print(A)

print("\nStandard deviation of all entries:")
print(torch.std(A))

print("\nStandard deviation along dim=0:")
print(torch.std(A, dim=0))

print("\nStandard deviation along dim=1:")
print(torch.std(A, dim=1))

A:
tensor([[1., 2., 3.],
        [4., 5., 6.]])

Standard deviation of all entries:
tensor(1.8708)

Standard deviation along dim=0:
tensor([2.1213, 2.1213, 2.1213])

Standard deviation along dim=1:
tensor([1., 1.])


## 4. Querying and conditionally transforming tensors

### Finding extrema with `torch.argmin()` and `torch.argmax()`

The functions `torch.argmin()` and `torch.argmax()` are used to find the **indices** of the smallest and largest values in a tensor.

In the 1D case:

- `torch.argmin(x)` returns the **position of the smallest value**
- `torch.argmax(x)` returns the **position of the largest value**

So for a 1D tensor, the result tells us *where* the extreme values occur.

---

In the 2D case:

- If no dimension is specified, PyTorch first **flattens the tensor into 1D**
- The returned index then corresponds to the position in this flattened tensor

This is why the output for `argmin(A)` and `argmax(A)` is a **single scalar index**, rather than a row/column pair.

---

**Key idea**:

- `argmin` and `argmax` return **indices**, not the values themselves
- They are commonly used to:
  - locate optimal values,
  - identify positions of interest,
  - and guide decision-making in algorithms

To obtain more structured results (e.g. row-wise or column-wise indices), we can specify a dimension using the `dim` argument (shown in the next section).

In [9]:
x = torch.tensor([2.5, -1.0, 3.2, 0.7, -4.1])

print("x:", x)
print("argmin:", torch.argmin(x))
print("argmax:", torch.argmax(x))

A = torch.tensor([
    [3.0, 8.0, 1.0],
    [5.0, 2.0, 9.0]
])

print("\n A:")
print(A)

print("argmin(A):", torch.argmin(A))
print("argmax(A):", torch.argmax(A))

x: tensor([ 2.5000, -1.0000,  3.2000,  0.7000, -4.1000])
argmin: tensor(4)
argmax: tensor(2)

 A:
tensor([[3., 8., 1.],
        [5., 2., 9.]])
argmin(A): tensor(2)
argmax(A): tensor(5)


In [10]:
print("argmin along dim=0:", torch.argmin(A, dim=0))
print("argmax along dim=0:", torch.argmax(A, dim=0))

print("argmin along dim=1:", torch.argmin(A, dim=1))
print("argmax along dim=1:", torch.argmax(A, dim=1))

argmin along dim=0: tensor([0, 1, 0])
argmax along dim=0: tensor([1, 0, 1])
argmin along dim=1: tensor([2, 1])
argmax along dim=1: tensor([1, 2])


### Conditional selection with `torch.where()`

The function `torch.where(condition, x, y)` selects values **element-wise** based on a logical condition:

- if the condition is `True`, take the value from `x`
- if the condition is `False`, take the value from `y`

In this example:

- the condition is `x > 0`
- positive entries are kept unchanged
- all other entries are replaced with `0`

So `torch.where()` acts like a tensor-wise **if–else operation**.

---

**Key idea**:

`torch.where()` is useful when we want to modify a tensor according to some rule without writing an explicit loop.

It is commonly used for:
- thresholding,
- masking,
- clipping values conditionally,
- and building piecewise-defined tensor operations

This makes it a very powerful tool for conditional tensor manipulation in PyTorch.

In [11]:
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])

result = torch.where(x > 0, x, torch.tensor(0.0))

print("x:")
print(x)

print("\nKeep positive values, replace others with 0:")
print(result)

x:
tensor([-2., -1.,  0.,  1.,  2.])

Keep positive values, replace others with 0:
tensor([0., 0., 0., 1., 2.])


## 🧭 Closing Remarks

In this tutorial, we stayed at the tensor level and expanded the basic PyTorch toolbox introduced previously.

The purpose was no longer only to recognise what a tensor is, but to become more fluent in the kinds of operations that are constantly used when working with tensors in practice:

- reshaping them,
- constructing useful structured tensors,
- combining them,
- querying them,
- transforming them element-wise,
- and summarising them statistically.

That was the central goal of the notebook.

At first sight, the operations introduced here may appear to be a miscellaneous collection of utility functions:

- `squeeze()` and `unsqueeze()`,
- `torch.linspace()` and `torch.eye()`,
- `torch.cat()`,
- `torch.argmin()` and `torch.argmax()`,
- `torch.where()`,
- and `torch.std()`.

But the point of the tutorial was precisely that these are not random commands.
They form part of the practical working language of tensor computation.

Once tensors are understood as structured mathematical objects, the next question naturally becomes:

- how can their shape be adapted,
- how can useful tensors be built efficiently,
- how can multiple tensors be joined together,
- how can important entries be located,
- and how can tensor values be transformed or summarised in a controlled way?

This notebook was designed to answer those questions.

That is why the emphasis remained on interpretation rather than memorisation.

For example, the important lesson behind `squeeze()` and `unsqueeze()` was not merely the syntax of the functions.
It was the idea that even when the underlying data are unchanged, **tensor shape still matters**, because shape determines how later operations interpret the data.

Similarly, `torch.cat()` was important not only because it joins tensors, but because it reinforces the idea that dimensions are meaningful:
concatenating along `dim=0` is not the same as concatenating along `dim=1`, and understanding that distinction is essential.

The same was true for the querying and reduction operations.

With `torch.argmin()` and `torch.argmax()`, the point was not only to find extrema, but to recognise that PyTorch often returns the **location** of information rather than only the information itself.

With `torch.std()`, the important idea was that a tensor can be reduced to a smaller object that captures a statistical summary of its entries.

And with `torch.where()`, we saw that conditional logic can also be expressed directly at the tensor level, without explicit Python loops.

Taken together, these operations illustrate an important shift in fluency.

In Tutorial 1, the main question was:

> what is a tensor, and how should it be interpreted?

Here, the question became:

> once I have a tensor, how do I manipulate it in ways that preserve clear mathematical meaning?

That is an important step.

It moves us from basic recognition of tensors toward practical competence in using them.

In that sense, this tutorial was still deliberately narrow in scope.
We did not yet discuss automatic differentiation, parameters, loss functions, or model training.
That was intentional.

Before tensors can participate in learning, it is important to be comfortable with the kinds of tensor operations that appear constantly inside real PyTorch code.

So by the end of this notebook, the important outcome is not merely that we have seen more PyTorch syntax.
It is that the tensor toolbox should now feel broader, more usable, and more coherent.

Tensors should now feel not only like objects that can be created and inspected, but like objects that can also be:

- reshaped with purpose,
- constructed with structure,
- combined meaningfully,
- queried precisely,
- and transformed systematically.

That gives us a natural bridge into the next stage of the series.

In **Tutorial 3**, we will begin moving beyond tensor operations alone and introduce how PyTorch tracks computations involving tensors so that gradients can be calculated automatically.
At that point, the tensor operations developed here will begin to play a deeper role inside computation graphs and learning workflows.

So the main takeaway of this tutorial is:

> understanding PyTorch requires not only knowing what tensors are, but also being able to **reshape, construct, combine, query, and transform them fluently**.